# <span style="color: #1F1DB5;">SPRINT 11 – Aprendizaje Supervisado: Optimización

# <span style="color: #1F1DB5;">Notebook 2 – Feature Engineering, Encoding y Selección de Características — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Aplicar One-Hot Encoding y Ordinal Encoding correctamente según el tipo de variable.
- Realizar Feature Engineering avanzado: binning, log transform, interacciones y extracción de fechas.
- Comprender la correlación lineal vs no-lineal y su impacto en modelos.
- Aplicar técnicas de selección de características: importancia, RFE, SelectKBest.
- Construir ColumnTransformer para pipelines con diferentes transformaciones por columna.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Aplicar **One-Hot Encoding** y **Ordinal Encoding** correctamente según el tipo de variable.
- Realizar **Feature Engineering** avanzado: binning, log transform, interacciones y extracción de fechas.
- Comprender la **correlación lineal vs no-lineal** y su impacto en modelos.
- Aplicar técnicas de **selección de características**: importancia, RFE, SelectKBest.
- Construir **ColumnTransformer** para pipelines con diferentes transformaciones por columna.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
OHE vs Ordinal Encoding: cuándo usar cada uno | 20 min |
Feature Engineering: crear mejores features | 25 min |
Correlación y selección de características | 20 min |
ColumnTransformer: pipelines profesionales | 15 min |
Actividad práctica – Breakout Rooms | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo podemos transformar los datos para que un modelo encuentre patrones que antes no podía ver?

La respuesta está en los datos. Un modelo de ML es tan bueno como los features que recibe. El **Feature Engineering** — el arte de transformar, crear y seleccionar características — es frecuentemente lo que separa un modelo promedio de uno excepcional. 

En competencias de Kaggle, los ganadores suelen decir: "80% del tiempo lo dedicamos a feature engineering, 20% a probar modelos."

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">OHE vs Ordinal Encoding: cuándo usar cada uno (20 mins)

Los modelos de ML trabajan con números, no con texto. Cuando tenemos variables categóricas (como "color", "ciudad", "nivel_educativo"), necesitamos convertirlas a formato numérico. Pero **la forma en que las codificamos importa mucho**.

### <span style="color: #2772F2;">One-Hot Encoding (OHE)

Crea una columna binaria (0/1) por cada categoría. Es ideal para variables **nominales** — donde no hay un orden natural entre las categorías.

**Usar OHE cuando:**
- Las categorías NO tienen orden (ej: rojo, azul, verde)
- Ciudades, países, tipos de producto, género
- El modelo es sensible a relaciones numéricas falsas (regresión lineal, KNN)

**Cuidado con:**
- Muchas categorías → muchas columnas (curse of dimensionality)
- Usar `drop='first'` para evitar multicolinealidad en regresión

### <span style="color: #2772F2;">Ordinal Encoding

Asigna un número entero a cada categoría. Es ideal para variables **ordinales** — donde hay un orden natural.

**Usar Ordinal cuando:**
- Las categorías TIENEN orden (ej: bajo, medio, alto)
- Nivel educativo, talla de ropa (S, M, L, XL), calificación (1-5 estrellas)
- Los árboles de decisión (que pueden aprovechar el orden)

**Error común:** Aplicar Ordinal Encoding a variables nominales — el modelo interpreta que "3 > 1", creando relaciones falsas.

Creemos un dataset mixto con variables nominales y ordinales para demostrar la diferencia entre ambos encodings.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Dataset de ejemplo: empleados
data = pd.DataFrame({
    'departamento': ['ventas', 'tech', 'rrhh', 'tech', 'ventas', 'rrhh', 'tech', 'ventas'],
    'nivel_experiencia': ['junior', 'senior', 'mid', 'junior', 'senior', 'mid', 'senior', 'junior'],
    'ciudad': ['CDMX', 'GDL', 'MTY', 'CDMX', 'GDL', 'CDMX', 'MTY', 'GDL'],
    'satisfaccion': ['baja', 'alta', 'media', 'media', 'alta', 'baja', 'alta', 'media'],
    'salario': [30000, 65000, 45000, 35000, 70000, 40000, 68000, 32000]
})

print("Dataset original:")
print(data)
print(f"\nTipos de variables categóricas:")
print(f"  - departamento: NOMINAL (no hay orden entre ventas, tech, rrhh)")
print(f"  - nivel_experiencia: ORDINAL (junior < mid < senior)")
print(f"  - ciudad: NOMINAL (no hay orden entre ciudades)")
print(f"  - satisfaccion: ORDINAL (baja < media < alta)")

In [ ]:
# One-Hot Encoding para variables NOMINALES
ohe = OneHotEncoder(sparse_output=False, drop='first')
nominales = ['departamento', 'ciudad']
X_ohe = ohe.fit_transform(data[nominales])
cols_ohe = ohe.get_feature_names_out(nominales)

print("=== One-Hot Encoding (variables nominales) ===")
print(pd.DataFrame(X_ohe, columns=cols_ohe))

# Ordinal Encoding para variables ORDINALES
oe = OrdinalEncoder(categories=[
    ['junior', 'mid', 'senior'],      # nivel_experiencia
    ['baja', 'media', 'alta']          # satisfaccion
])
ordinales = ['nivel_experiencia', 'satisfaccion']
X_oe = oe.fit_transform(data[ordinales])

print("\n=== Ordinal Encoding (variables ordinales) ===")
print(pd.DataFrame(X_oe, columns=ordinales))
print("\nNota: junior=0, mid=1, senior=2 → respeta el orden natural")

Preguntas:

- ¿Qué pasaría si aplicamos Ordinal Encoding a "departamento"? ¿Qué relación falsa interpretaría el modelo?

- ¿Por qué usamos drop='first' en OHE? ¿Qué problema resuelve?

- ¿Qué encoding usarías para una columna "talla" con valores XS, S, M, L, XL?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Feature Engineering: crear mejores features (25 mins)

Feature Engineering es el proceso de **crear nuevas variables** a partir de las existentes para darle al modelo información más útil. Es donde la creatividad y el conocimiento del dominio marcan la diferencia.

### <span style="color: #2772F2;">Técnicas principales:

1. **Binning (discretización):** Convertir una variable continua en categorías
   - Edad → [joven, adulto, senior]
   - Ingreso → [bajo, medio, alto]

2. **Log Transform:** Reducir el sesgo (skewness) en distribuciones con cola larga
   - Precios, ingresos, conteos → `np.log1p(x)`

3. **Extracción de fechas:** Sacar información temporal de timestamps
   - Día de la semana, mes, hora, ¿es fin de semana?

4. **Ratios e interacciones:** Crear proporciones o multiplicaciones entre features
   - ingreso/gastos, precio/m², horas_estudio * dificultad

5. **Agregaciones:** Estadísticas grupales
   - promedio_ventas_por_región, max_compras_por_cliente

Vamos a crear un dataset más realista y demostrar cómo el Feature Engineering puede mejorar dramáticamente el rendimiento de un modelo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Crear dataset simulando datos de un e-commerce
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'precio': np.random.exponential(500, n) + 50,
    'cantidad': np.random.poisson(3, n) + 1,
    'hora_compra': np.random.randint(0, 24, n),
    'dia_semana': np.random.randint(0, 7, n),
    'antiguedad_cliente_dias': np.random.exponential(365, n),
    'num_compras_previas': np.random.poisson(5, n),
    'descuento_pct': np.random.uniform(0, 50, n),
})

# Target: si la compra será devuelta (más probable con precio alto, cliente nuevo, fin de semana)
prob_devolucion = (
    0.1 + 
    0.3 * (df['precio'] > 800).astype(float) +
    0.2 * (df['antiguedad_cliente_dias'] < 90).astype(float) +
    0.15 * (df['dia_semana'] >= 5).astype(float)
)
df['devolucion'] = (np.random.random(n) < prob_devolucion).astype(int)

print(f"Dataset: {df.shape[0]} compras, {df.shape[1]-1} features")
print(f"Tasa de devolución: {df['devolucion'].mean()*100:.1f}%")
print(f"\nPrimeras filas:")
print(df.head())

In [ ]:
# Modelo BASELINE sin Feature Engineering
X_base = df.drop('devolucion', axis=1)
y = df['devolucion']

rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
scores_base = cross_val_score(rf_base, X_base, y, cv=5, scoring='f1')
print(f"=== Modelo BASELINE (sin FE) ===")
print(f"  F1-score (CV): {scores_base.mean():.4f} ± {scores_base.std():.4f}")

In [ ]:
# Ahora apliquemos Feature Engineering
df_fe = df.copy()

# 1. Log transform en precio (distribución sesgada)
df_fe['log_precio'] = np.log1p(df_fe['precio'])

# 2. Binning de hora en franjas
df_fe['franja_horaria'] = pd.cut(df_fe['hora_compra'], 
    bins=[0, 6, 12, 18, 24], labels=[0, 1, 2, 3]).astype(int)

# 3. Es fin de semana (extracción de fecha)
df_fe['es_finde'] = (df_fe['dia_semana'] >= 5).astype(int)

# 4. Ratio: gasto total / antiguedad (intensidad de compra)
df_fe['gasto_total'] = df_fe['precio'] * df_fe['cantidad']
df_fe['intensidad_compra'] = df_fe['gasto_total'] / (df_fe['antiguedad_cliente_dias'] + 1)

# 5. Cliente nuevo (menos de 90 días)
df_fe['cliente_nuevo'] = (df_fe['antiguedad_cliente_dias'] < 90).astype(int)

# 6. Descuento real aplicado
df_fe['ahorro'] = df_fe['precio'] * df_fe['descuento_pct'] / 100

print("Nuevos features creados:")
print(df_fe[['log_precio', 'franja_horaria', 'es_finde', 
             'gasto_total', 'intensidad_compra', 'cliente_nuevo', 'ahorro']].head())

In [ ]:
# Modelo CON Feature Engineering
X_fe = df_fe.drop('devolucion', axis=1)

rf_fe = RandomForestClassifier(n_estimators=100, random_state=42)
scores_fe = cross_val_score(rf_fe, X_fe, y, cv=5, scoring='f1')
print(f"=== Modelo CON Feature Engineering ===")
print(f"  F1-score (CV): {scores_fe.mean():.4f} ± {scores_fe.std():.4f}")
print(f"\n  Mejora vs baseline: +{(scores_fe.mean() - scores_base.mean())*100:.2f} puntos porcentuales")
print(f"  Mejora relativa: +{((scores_fe.mean()/scores_base.mean())-1)*100:.1f}%")

Preguntas:

- ¿Cuál de los nuevos features crees que aporta más información al modelo? ¿Por qué?

- ¿Se te ocurre algún otro feature que podríamos crear con las columnas disponibles?

- ¿Por qué el log transform ayuda con variables que tienen distribución sesgada?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Correlación y selección de características (20 mins)

### <span style="color: #2772F2;">Correlación: Pearson vs no-lineal

**Correlación de Pearson** mide relaciones **lineales** entre dos variables. Su valor va de -1 a +1:
- +1: correlación positiva perfecta (ambas suben juntas)
- -1: correlación negativa perfecta (una sube, otra baja)
- 0: sin correlación lineal (¡pero puede haber relación no-lineal!)

**Limitación importante:** Pearson NO detecta relaciones cuadráticas, exponenciales, o cualquier patrón no-lineal. Dos variables pueden tener Pearson ≈ 0 y aún así estar fuertemente relacionadas.

### <span style="color: #2772F2;">Selección de características

No todos los features son útiles. Algunos:
- Son redundantes (alta correlación entre sí)
- Son irrelevantes (no aportan información al target)
- Añaden ruido que confunde al modelo

**Técnicas de selección:**
1. **feature_importances_** (Random Forest): mide cuánto contribuye cada feature
2. **SelectKBest**: selecciona los k mejores features según un test estadístico
3. **RFE (Recursive Feature Elimination)**: elimina features iterativamente, quedándose con los más importantes

Visualicemos la correlación y luego apliquemos técnicas de selección para identificar los features más relevantes.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE

# Importancia de features con Random Forest
rf_fe.fit(X_fe, y)
importances = pd.Series(rf_fe.feature_importances_, index=X_fe.columns)
importances_sorted = importances.sort_values(ascending=True)

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
importances_sorted.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importancia')
ax.set_title('Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

print("\nTop 5 features más importantes:")
for feat, imp in importances.sort_values(ascending=False).head().items():
    print(f"  {feat}: {imp:.4f}")

In [ ]:
# SelectKBest: seleccionar los 5 mejores features
selector = SelectKBest(f_classif, k=5)
X_selected = selector.fit_transform(X_fe, y)

# Ver cuáles fueron seleccionados
mask = selector.get_support()
selected_features = X_fe.columns[mask].tolist()
print(f"=== SelectKBest (k=5) ===")
print(f"Features seleccionados: {selected_features}")
print(f"\nScores de cada feature:")
for feat, score in sorted(zip(X_fe.columns, selector.scores_), key=lambda x: -x[1]):
    marker = " ✅" if feat in selected_features else ""
    print(f"  {feat}: {score:.2f}{marker}")

Preguntas:

- ¿Los features que creamos con FE aparecen entre los más importantes? ¿Qué nos dice eso?

- ¿Cuándo preferirías usar RFE sobre SelectKBest?

### Mis respuestas de validación

- 
- 


In [ ]:
# Correlación de Pearson
import matplotlib.pyplot as plt

# Matriz de correlación del dataset con FE
corr_matrix = X_fe.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
plt.colorbar(im, ax=ax, label='Correlación de Pearson')
ax.set_title('Matriz de correlación - Features originales + engineered')
plt.tight_layout()
plt.show()

# Detectar features altamente correlacionados
print("\nPares con |correlación| > 0.8 (posible redundancia):")
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            print(f"  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_matrix.iloc[i,j]:.3f}")

### <span style="color: #2772F2;">La limitación de Pearson: relaciones no-lineales

Pearson solo captura relaciones lineales. Dos variables pueden tener correlación de Pearson ≈ 0 y aún así estar perfectamente relacionadas de forma no-lineal (cuadrática, logarítmica, etc.).

Por eso los árboles de decisión no dependen de Pearson — hacen cortes que capturan cualquier patrón. Pero para regresión lineal o KNN, Pearson sí es un buen indicador de qué features serán útiles.

## <span style="color: #2826DE;">ColumnTransformer: pipelines profesionales (15 mins)

En la práctica, un dataset tiene **columnas de diferentes tipos** que necesitan **transformaciones diferentes**:
- Numéricas → escalar (StandardScaler)
- Categóricas nominales → One-Hot Encoding
- Categóricas ordinales → Ordinal Encoding

El `ColumnTransformer` de sklearn nos permite aplicar **diferentes transformadores a diferentes columnas** dentro de un mismo pipeline. Esto es fundamental para:
- Mantener el código limpio y reproducible
- Evitar data leakage (todo se ajusta solo con train)
- Desplegar el modelo en producción con un solo objeto

Construyamos un pipeline completo que combine ColumnTransformer con un modelo, aplicando transformaciones apropiadas a cada tipo de columna.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import GradientBoostingClassifier

# Recrear dataset con columnas mixtas
df_mixed = pd.DataFrame({
    'edad': np.random.randint(18, 65, 200),
    'ingreso': np.random.exponential(40000, 200) + 15000,
    'ciudad': np.random.choice(['CDMX', 'GDL', 'MTY', 'PUE'], 200),
    'nivel_edu': np.random.choice(['preparatoria', 'licenciatura', 'maestria'], 200),
    'compras_mes': np.random.poisson(4, 200),
})
y_mixed = (df_mixed['ingreso'] > 45000).astype(int)  # Target simple

# Definir columnas por tipo
num_cols = ['edad', 'ingreso', 'compras_mes']
cat_nominal_cols = ['ciudad']
cat_ordinal_cols = ['nivel_edu']

# Crear ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat_nom', OneHotEncoder(drop='first', sparse_output=False), cat_nominal_cols),
    ('cat_ord', OrdinalEncoder(categories=[['preparatoria', 'licenciatura', 'maestria']]), cat_ordinal_cols)
])

# Pipeline completo: preprocesamiento + modelo
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

# Entrenar y evaluar
scores = cross_val_score(pipeline, df_mixed, y_mixed, cv=5, scoring='f1')
print(f"=== Pipeline con ColumnTransformer ===")
print(f"  F1-score (CV): {scores.mean():.4f} ± {scores.std():.4f}")
print(f"\nEstructura del pipeline:")
print(pipeline)

## <span style="color: #2826DE;">Tips y buenas prácticas

- 🎯 **Conoce tu dominio**: el mejor FE viene del conocimiento del negocio, no de fórmulas genéricas.
- 📐 **Log transform**: úsalo en features con distribución muy sesgada (precios, conteos, ingresos).
- 🗂️ **No crear demasiados features**: más features no siempre es mejor (curse of dimensionality).
- 🔍 **Valida con cross-validation**: cada feature nuevo debe demostrar que mejora el modelo.
- 🧹 **Elimina features redundantes**: alta correlación entre features → información duplicada.
- ⚙️ **ColumnTransformer es tu amigo**: nunca apliques el mismo encoding a todas las columnas.

## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

**Ejercicio: Mejorar un modelo con Feature Engineering**

Usando el dataset `df` que creamos anteriormente:

1. Crea al menos 3 nuevos features que creas relevantes para predecir devoluciones.
2. Construye un pipeline con ColumnTransformer que integre tus transformaciones.
3. Compara el F1-score antes y después de tus features.

**Pistas:**
- ¿Hay alguna interacción entre precio y descuento que sea informativa?
- ¿La hora de compra podría agruparse de forma más inteligente?
- ¿Qué pasa si creas un flag para compras "nocturnas"?

Al regresar, compartirás: qué features creaste y cuánto mejoró tu modelo.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Cuándo se debe usar One-Hot Encoding?

- Cuando las categorías tienen un orden natural
- Cuando las categorías son nominales (sin orden) 
- Solo cuando hay más de 10 categorías
- Solo para variables numéricas


2️⃣ ¿Qué hace np.log1p(x)?

- Calcula la raíz cuadrada de x
- Calcula log(x + 1), útil para reducir sesgo en distribuciones 
- Normaliza x entre 0 y 1
- Eleva x a la potencia e


3️⃣ ¿Qué mide la correlación de Pearson?

- Cualquier tipo de relación entre variables
- Solo relaciones lineales entre dos variables 
- La importancia de un feature para el modelo
- La distancia entre dos puntos


4️⃣ ¿Qué hace SelectKBest?

- Selecciona las k muestras más representativas
- Elimina las k columnas más correlacionadas
- Selecciona los k features con mejor score estadístico 
- Aplica PCA para reducir a k dimensiones


5️⃣ ¿Para qué sirve ColumnTransformer?

- Para transformar filas en columnas
- Para aplicar diferentes transformaciones a diferentes columnas 
- Para eliminar columnas con valores nulos
- Para unir dos DataFrames por columnas


6️⃣ ¿Cuál es un error común con Ordinal Encoding?

- Aplicarlo a variables con más de 5 categorías
- Usarlo sin StandardScaler
- Aplicarlo a variables nominales, creando un orden falso 
- Usarlo con Random Forest